In [ ]:
# Solutions live in solutions/. Run from the repository root so imports and paths work.
import os, pathlib, sys
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "workshop").is_dir())
os.chdir(_root)
sys.path.insert(0, str(_root))

# Station E: Lonboard

**Good for:** fast maps of large geospatial data (millions of points, lines,
polygons) from GeoPandas, using WebGL. By Development Seed.
**Internet:** only for the base map. With `basemap=None` you still see the data.

Docs: https://developmentseed.org/lonboard

In [ ]:
import geopandas as gpd
import ipywidgets as widgets
import lonboard
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Approximate city centres; sensors are scattered around each one.
cities = pd.DataFrame({
    "city": ["Delhi", "Mumbai", "Bengaluru", "Chennai", "Kolkata", "Hyderabad", "Pune", "Ahmedabad", "Jaipur", "Lucknow"],
    "lat": [28.61, 19.08, 12.97, 13.08, 22.57, 17.39, 18.52, 23.02, 26.91, 26.85],
    "lon": [77.21, 72.88, 77.59, 80.27, 88.36, 78.49, 73.86, 72.57, 75.79, 80.95],
})
rng = np.random.default_rng(7)
per_city = 5_000
sensors = cities.loc[cities.index.repeat(per_city)].reset_index(drop=True)
sensors["lat"] += rng.normal(0, 0.15, len(sensors))
sensors["lon"] += rng.normal(0, 0.15, len(sensors))
sensors["pm25"] = rng.gamma(4, 15, len(sensors)).round(1)

gdf = gpd.GeoDataFrame(sensors, geometry=gpd.points_from_xy(sensors.lon, sensors.lat), crs="EPSG:4326")
f"{len(gdf):,} air-quality sensors"

Colour by PM2.5 with a Matplotlib colormap, then draw the map. Click a point to pick it.

In [ ]:
norm = plt.Normalize(0, 150)
colors = (plt.cm.magma_r(norm(gdf["pm25"]))[:, :3] * 255).astype(np.uint8)

layer = lonboard.ScatterplotLayer.from_geopandas(
    gdf, get_fill_color=colors, get_radius=800, radius_min_pixels=2, pickable=True,
)
m = lonboard.Map(layers=[layer])  # offline? use lonboard.Map(layers=[layer], basemap=None)
m

### Your task: connect it

1. Show the clicked sensor's row in an `Output` (`layer.selected_index` is a synced trait).
2. Add a slider that hides sensors below a PM2.5 threshold. Hint: rebuild the
   colours with alpha (RGBA, 4 columns) and set `layer.get_fill_color`.

In [ ]:
picked = widgets.Output()
threshold = widgets.IntSlider(value=0, min=0, max=150, step=5, description="PM2.5 ≥")


def on_pick(change):
    index = change["new"]
    with picked:
        picked.clear_output(wait=True)
        if index is not None:
            display(gdf.drop(columns="geometry").iloc[[index]])


def on_threshold(change):
    rgba = (plt.cm.magma_r(norm(gdf["pm25"])) * 255).astype(np.uint8)
    rgba[gdf["pm25"].to_numpy() < change["new"], 3] = 0
    layer.get_fill_color = rgba


layer.observe(on_pick, names="selected_index")
threshold.observe(on_threshold, names="value")
widgets.VBox([threshold, picked])